# PiggyTech Sentiment Analysis — Local / VS Code Edition

Ported from the Colab notebook (`PiggyTech_Sentiment_Analysis_colab_original.ipynb` — kept in this folder for reference).

**Key differences from the Colab version**
- No Google Drive mount. All artefacts live under `ml/data/` and `ml/models/` in this repo.
- No `google.colab.userdata`. Secrets come from `ml/.env` via `python-dotenv`.
- Auto-detects device: **MPS** on Apple Silicon, CUDA if available, else CPU.
- Dependencies come from `ml/requirements.txt` — **don't** run `pip install` inside the notebook.
- **Dashboard-ready timestamps**: every scraped tweet gets a full suite of time columns (UTC + Lagos local, date, hour-of-day, day-of-week, ISO week) derived once at scrape time, so Looker Studio can filter cleanly without runtime parsing.

**Timestamp columns written to every CSV**

| Column | Format | Use |
|---|---|---|
| `posted_at_utc` | `2025-09-15T14:32:07Z` (ISO 8601) | Canonical — unambiguous UTC |
| `posted_at_lagos` | `2025-09-15T15:32:07+01:00` | Lagos local, for user-facing "when did this happen" |
| `posted_date` | `2025-09-15` | Lagos-local day for daily filters |
| `posted_hour_utc` | `14` | UTC hour (0-23) |
| `posted_hour_lagos` | `15` | Lagos hour — for "peak complaint windows" |
| `posted_dow` | `Monday` | Day-of-week name |
| `posted_week` | `2025-W37` | ISO week — for weekly trend charts |
| `created_at` | `2025-09-15T14:32:07Z` | Legacy alias of `posted_at_utc` (kept for old dashboard queries) |
| `collected_at` | ISO-ish | When Scweet scraped it (not when posted) |

> **Before running:** see `ml/README.md` for one-time setup (venv, kernel, secrets, bringing over your Drive CSVs).

---

## Pipeline
| Step | Description |
|------|-------------|
| 1 | Imports & environment |
| 2 | Paths, secrets, device, API clients |
| 3 | Schema & taxonomy |
| 4 | Scrape tweets (Scweet via Apify) |
| 5 | Clean & filter |
| 6 | Auto-label with Gemini |
| 7 | Quality check & train/test split |
| 8 | Fine-tune classification model |
| 9 | Fine-tune regression model |
| 10 | (Optional) export for Label Studio |


## Step 1: Imports

In [19]:
# NOTE: install dependencies *outside* the notebook with:
#   python3 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt
# Do NOT run `!pip install` inside the notebook — it can install into the wrong env.

import os
import re
import json
import time
import random
from datetime import date, datetime, timezone, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from apify_client import ApifyClient
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, mean_squared_error,
)
import google.generativeai as genai

import torch

# Python 3.14 changed pickle._Pickler._batch_setitems to accept (items, obj).
# dill (used by HuggingFace datasets for fingerprinting) hasn't been updated yet.
# Patch generate_fingerprint so a TypeError falls back to a random fingerprint.
try:
    import datasets.fingerprint as _hf_fp
    _orig_gen_fp = _hf_fp.generate_fingerprint
    def _safe_generate_fingerprint(dataset):
        try:
            return _orig_gen_fp(dataset)
        except TypeError:
            import hashlib, uuid
            return hashlib.md5(str(uuid.uuid4()).encode()).hexdigest()
    _hf_fp.generate_fingerprint = _safe_generate_fingerprint
    print('🔧 Python 3.14 datasets fingerprint patch applied')
except ImportError:
    pass  # datasets not installed yet — patch will be applied when Step 8 imports it

print('✅ Imports loaded')


✅ Imports loaded


## Step 2: Paths, secrets, device, API clients

- `DATA_DIR` holds the CSVs (raw / filtered / labelled).
- `MODEL_DIR` holds the fine-tuned model weights + tokenizer.
- `CHECKPOINT_DIR` holds intermediate training checkpoints.

If you have existing CSVs from Drive (`raw_tweets_all.csv`, `filtered_tweets_all.csv`, `labelled_tweets_unified.csv`), drop them into `ml/data/` before running the labelling/training cells — the notebook will pick up where Colab left off.

In [20]:
# ── paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()                   # = .../piggytech-sentiment/ml
DATA_DIR       = NOTEBOOK_DIR / 'data'
MODEL_DIR      = NOTEBOOK_DIR / 'models'
CHECKPOINT_DIR = NOTEBOOK_DIR / 'checkpoints'
for p in (DATA_DIR, MODEL_DIR, CHECKPOINT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── secrets from ml/.env (see ml/.env.example) ─────────────────────────────
load_dotenv(NOTEBOOK_DIR / '.env')

APIFY_API_TOKEN = os.environ.get('APIFY_API_TOKEN')
GEMINI_API_KEY  = os.environ.get('GEMINI_API_KEY')

if not APIFY_API_TOKEN:
    print('⚠️  APIFY_API_TOKEN not set — Step 4 (scraping) will fail until you add it to ml/.env')
if not GEMINI_API_KEY:
    print('⚠️  GEMINI_API_KEY not set  — Step 6 (labelling) will fail until you add it to ml/.env')

# ── API clients ────────────────────────────────────────────────────────────
apify_client = ApifyClient(APIFY_API_TOKEN) if APIFY_API_TOKEN else None

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    gemini = genai.GenerativeModel('gemini-2.5-flash')
else:
    gemini = None

# ── device: MPS (Apple Silicon) > CUDA > CPU ───────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'📁 Data dir:       {DATA_DIR}')
print(f'📁 Model dir:      {MODEL_DIR}')
print(f'📁 Checkpoint dir: {CHECKPOINT_DIR}')
print(f'🖥  Torch device:   {DEVICE}')
print(f'✅ API clients: Apify={"ready" if apify_client else "missing token"}, '
      f'Gemini={"ready" if gemini else "missing token"}')


📁 Data dir:       /Users/honoredone/piggytech-sentiment/ml/data
📁 Model dir:      /Users/honoredone/piggytech-sentiment/ml/models
📁 Checkpoint dir: /Users/honoredone/piggytech-sentiment/ml/checkpoints
🖥  Torch device:   mps
✅ API clients: Apify=ready, Gemini=ready


## Step 3: Schema & taxonomy

In [21]:
# Sentiment labels (5-point scale)
sentiment_labels = [
    'very_negative',
    'slightly_negative',
    'neutral',
    'slightly_positive',
    'very_positive',
]

# Intent labels — used as aspect categories, not for filtering
intent_labels = ['opinion', 'complaint', 'inquiry', 'suggestion', 'spam']

# Products
products = ['piggyvest', 'piggyvest for business', 'pocket']

# Aspects per product (v2: inquiry and suggestion added as shared aspects)
aspects = {
    'shared': [
        'customer_support', 'app_performance',
        'payments_transfers', 'notifications', 'trust_security', 'brand_perception',
        'inquiry', 'suggestion',
    ],
    'PiggyVest': ['savings_plans', 'withdrawals', 'investments', 'savings_report'],
    'PiggyVest_for_Business': ['merchant_onboarding', 'settlements', 'collections'],
    'Pocket': ['checkout_payments', 'wallet_funding', 'transfers'],
}

# Product aliases (used by filtering + Gemini prompt)
product_aliases = {
    'PiggyVest': [
        'piggyvest', 'piggy vest', 'piggy', '@PiggyVest',
        'piggyvest app', 'piggyvestapp', 'safelock', 'safe lock',
        'piggybank', 'autosave', 'target savings', 'flex naira',
        'flex dollar', 'investify', 'piggypoints', 'savings_report',
        'piggyvest report', 'savings summary',
    ],
    'PiggyVest_for_Business': [
        'pvb', 'piggyvest for business', 'piggyvest business',
        'piggytech business', '@PVB', 'business safelock',
        'bulk transfer', 'salary disbursement',
    ],
    'Pocket': [
        'pocket app', 'pocket by piggyvest', '@usepocket',
        'usepocket', 'pocket payment', 'pocket wallet',
        'pocket transfer', 'pocket bill', 'payment link',
    ],
}

# Dataset schema — what every row in the labelled CSV looks like
schema = {
    'tweet_id': '',
    'tweet_text': '',
    # Dashboard timestamp suite — all second-precision, all derived from the tweet's snowflake ID
    # or Scweet's native date field (whichever is present)
    'posted_at_utc': '',       # '2025-09-15T14:32:07Z'   canonical UTC (ISO 8601 with Z)
    'posted_at_lagos': '',     # '2025-09-15T15:32:07+01:00'  Lagos WAT
    'posted_date': '',         # '2025-09-15'  Lagos-local day for filters
    'posted_hour_utc': 0,      # 14
    'posted_hour_lagos': 0,    # 15 — peak-complaint-hour charts use this
    'posted_dow': '',          # 'Monday'
    'posted_week': '',         # '2025-W37'
    'created_at': '',          # legacy alias of posted_at_utc (kept for old queries)
    'collected_at': '',        # when Scweet scraped it, NOT when posted
    # Author
    'author_username': '',
    'author_name': '',
    'author_followers': 0,
    # Labels (filled by Gemini in Step 6)
    'products_detected': [],
    'overall_sentiment': '',
    'intent': '',
    'aspect_product': '',
    'aspect': '',
    'aspect_sentiment': '',
}

print('✅ Schema and taxonomy defined')
print(f'   Aspects (shared): {aspects["shared"]}')
print(f'   Intent labels:    {intent_labels}')


✅ Schema and taxonomy defined
   Aspects (shared): ['customer_support', 'app_performance', 'payments_transfers', 'notifications', 'trust_security', 'brand_perception', 'inquiry', 'suggestion']
   Intent labels:    ['opinion', 'complaint', 'inquiry', 'suggestion', 'spam']


In [22]:
# Row validator
def validate_row(row: dict) -> bool:
    valid_products   = set(product_aliases.keys())
    valid_sentiments = set(sentiment_labels)
    valid_intents    = set(intent_labels)
    all_aspects = (
        aspects['shared'] + aspects['PiggyVest']
        + aspects['PiggyVest_for_Business'] + aspects['Pocket']
    )
    errors = []

    if row.get('overall_sentiment') and row['overall_sentiment'] not in valid_sentiments:
        errors.append(f"Invalid overall_sentiment: {row['overall_sentiment']}")
    if row.get('intent') and row['intent'] not in valid_intents:
        errors.append(f"Invalid intent: {row['intent']}")
    for p in (row.get('products_detected') or []):
        if p not in valid_products:
            errors.append(f'Unknown product: {p}')
    if row.get('aspect_product') and row['aspect_product'] not in valid_products:
        errors.append(f"Invalid aspect_product: {row['aspect_product']}")
    if row.get('aspect') and row['aspect'] not in all_aspects:
        errors.append(f"Unknown aspect: {row['aspect']}")
    if row.get('aspect_sentiment') and row['aspect_sentiment'] not in valid_sentiments:
        errors.append(f"Invalid aspect_sentiment: {row['aspect_sentiment']}")

    if errors:
        print(f"❌ Row {row.get('tweet_id', '?')} errors: {errors}")
        return False
    return True

print('✅ validate_row() ready')


✅ validate_row() ready


## Step 4: Scrape tweets — maximum-volume strategy

Free Scweet plan: **1,000 tweets/day, 10 runs/day, min 60s between runs**.
Strategies run: Search-Latest ✕ 2 date ranges → Search-Top → profile timelines.
All merged + deduped into `data/raw_tweets_all.csv` (APPEND — safe to re-run).

In [23]:
# Brand keywords for search queries and filtering
brand_keywords = [
    'piggyvest', 'piggy vest', '@piggyvest',
    'pocket app', '@usepocket', 'usepocket',
    'pocket by piggyvest', 'pvb', 'piggyvest for business',
    'investify', 'piggypoints',
    'flex dollar piggyvest', 'flex naira piggyvest',
    'savings report', 'piggyvest report',
    'piggyvest savings report', 'savings summary',
]

# Official product accounts
product_profiles = ['@PiggyVest', '@usepocket']


# Nigeria uses WAT (UTC+1, no DST). Dashboard users think in Lagos time.
LAGOS_TZ = timezone(timedelta(hours=1))

# Fields Scweet may use for the tweet's posted-at timestamp, in preference order.
# Different Scweet actor versions rename these; we try them all.
TWEET_DATE_FIELDS = (
    'posted_at_utc', 'datetime', 'date', 'created_at', 'timestamp', 'posted_at',
)


def tweet_id_to_datetime_utc(tweet_id):
    """Decode a Twitter snowflake ID to a UTC-aware datetime (second precision).
    Returns None when the ID isn't a valid snowflake."""
    try:
        clean_id = str(tweet_id).replace('tweet-', '').strip()
        if not clean_id.isdigit() or len(clean_id) < 10:
            return None
        timestamp_ms = (int(clean_id) >> 22) + 1288834974657
        return datetime.fromtimestamp(timestamp_ms / 1000, tz=timezone.utc)
    except Exception:
        return None


def _parse_iso_utc(value):
    """Parse an ISO-8601-ish string to a UTC-aware datetime. Returns None on failure."""
    if not value or not isinstance(value, str):
        return None
    s = value.strip().replace('Z', '+00:00')
    dt = None
    try:
        dt = datetime.fromisoformat(s)
    except ValueError:
        for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%dT%H:%M:%S', '%a %b %d %H:%M:%S %z %Y'):
            try:
                dt = datetime.strptime(s, fmt)
                break
            except ValueError:
                continue
    if dt is None:
        return None
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def extract_posted_at(item, tweet_id):
    """Get the tweet's creation time as a UTC-aware datetime down to the second.
    Tries Scweet's native date fields first, falls back to snowflake ID decode.
    Returns None only if both strategies fail — which should be rare."""
    for key in TWEET_DATE_FIELDS:
        dt = _parse_iso_utc(item.get(key))
        if dt is not None:
            return dt
    return tweet_id_to_datetime_utc(tweet_id)


def timestamp_columns(posted_at_utc_dt):
    """Given a UTC datetime (or None), build the full set of dashboard columns.
    All derived from the same source so filter dimensions never disagree."""
    if posted_at_utc_dt is None:
        return {
            'posted_at_utc': '', 'posted_at_lagos': '',
            'posted_date': '', 'posted_hour_utc': '',
            'posted_hour_lagos': '', 'posted_dow': '', 'posted_week': '',
        }
    utc = posted_at_utc_dt
    lagos = utc.astimezone(LAGOS_TZ)
    return {
        # Canonical: ISO-8601 UTC with Z — unambiguous in every BI tool
        'posted_at_utc':    utc.strftime('%Y-%m-%dT%H:%M:%SZ'),
        # Lagos local ISO-8601 with offset — "when did this actually happen locally?"
        'posted_at_lagos':  lagos.strftime('%Y-%m-%dT%H:%M:%S+01:00'),
        # Pre-derived filter keys (Lagos-local so days line up with Nigerian users)
        'posted_date':      lagos.strftime('%Y-%m-%d'),
        'posted_hour_utc':  utc.hour,
        'posted_hour_lagos': lagos.hour,      # "peak complaint hour" lives here
        'posted_dow':       lagos.strftime('%A'),    # Monday, Tuesday, …
        'posted_week':      lagos.strftime('%Y-W%V'),  # ISO week (e.g. 2025-W37)
    }


def build_search_query(extra_terms=None):
    core_keywords = [
        'piggyvest', 'piggy vest', '@PiggyVest',
        'safelock', 'investify', 'piggypoints',
        'pocket app', '@usepocket', 'usepocket',
        'pvb', 'piggyvest for business',
        'piggyvest report', 'savings report',
        'piggy savings', 'piggy app',
    ]
    if extra_terms:
        core_keywords.extend(extra_terms)
    or_clause = ' OR '.join(f'"{kw}"' if ' ' in kw else kw for kw in core_keywords)
    return f'({or_clause}) -filter:retweets lang:en'


def _tweet_from_item(item, scrape_source):
    user = item.get('user', {}) or {}
    tweet_id = str(item.get('id', ''))
    ts = timestamp_columns(extract_posted_at(item, tweet_id))
    return {
        'tweet_id': tweet_id,
        'tweet_text': item.get('text', ''),
        # ── full timestamp suite (all derived from the SAME source) ──────
        **ts,
        # legacy alias — keeps old dashboard queries / Looker Studio fields working
        'created_at': ts['posted_at_utc'],
        'collected_at': item.get('collected_at_utc', ''),
        # ── author ───────────────────────────────────────────────────────
        'author_username': item.get('handle', '') or user.get('handle', ''),
        'author_name': user.get('name', ''),
        'author_followers': user.get('followers_count', 0),
        # ── engagement ───────────────────────────────────────────────────
        'favorite_count': item.get('favorite_count', 0),
        'retweet_count': item.get('retweet_count', 0),
        'reply_count': item.get('reply_count', 0),
        'tweet_url': item.get('tweet_url', ''),
        'scrape_source': scrape_source,
    }


def run_search_scrape(label, since, until, sort='Latest', max_items=100_000):
    assert apify_client is not None, 'APIFY_API_TOKEN missing — set it in ml/.env'
    query = build_search_query()
    run_input = {
        'source_mode': 'search',
        'search_query': query,
        'since': since,
        'until': until,
        'search_sort': sort,
        'max_items': max_items,
    }
    print(f'🔍 [{label}] Search ({sort}) from {since} to {until}...')
    run = apify_client.actor('altimis/scweet').call(run_input=run_input)
    tweets = [
        _tweet_from_item(item, f'search_{sort.lower()}')
        for item in apify_client.dataset(run['defaultDatasetId']).iterate_items()
    ]
    print(f'  ✅ Got {len(tweets)} tweets')
    return tweets


def run_profile_scrape(label, profiles, max_items=100_000):
    assert apify_client is not None, 'APIFY_API_TOKEN missing — set it in ml/.env'
    run_input = {
        'source_mode': 'profiles',
        'profile_urls': profiles,
        'max_items': max_items,
    }
    print(f'👤 [{label}] Profile timelines: {profiles}...')
    run = apify_client.actor('altimis/scweet').call(run_input=run_input)
    tweets = [
        _tweet_from_item(item, 'profile_timeline')
        for item in apify_client.dataset(run['defaultDatasetId']).iterate_items()
    ]
    print(f'  ✅ Got {len(tweets)} tweets')
    return tweets

print('✅ Scraping functions ready')
print(f'   Query preview: {build_search_query()[:80]}...')


✅ Scraping functions ready
   Query preview: (piggyvest OR "piggy vest" OR @PiggyVest OR safelock OR investify OR piggypoints...


In [24]:
# =============================================================
# STRATEGY 1: Search (Latest) — dynamic quarterly windows to today
# =============================================================
all_raw_tweets = []
today = date.today()

# Detect last date already saved so we don't re-scrape it
last_date = None
raw_csv_path = DATA_DIR / 'raw_tweets_all.csv'
if raw_csv_path.exists():
    _ex = pd.read_csv(raw_csv_path)
    if 'posted_date' in _ex.columns:
        _dates = _ex['posted_date'].dropna().astype(str)
        _dates = _dates[_dates.str.match(r'\d{4}-\d{2}-\d{2}')]
        if not _dates.empty:
            last_date = datetime.strptime(_dates.max(), '%Y-%m-%d').date()

DEFAULT_START = date(2025, 4, 1)
window_start = (last_date + timedelta(days=1)) if last_date else DEFAULT_START

if window_start > today:
    print(f'✅ Already up to date — last scraped date: {last_date}')
else:
    # Build calendar-quarter windows between window_start and today
    def _quarter_end(d):
        if d.month <= 3:  return date(d.year, 3, 31)
        if d.month <= 6:  return date(d.year, 6, 30)
        if d.month <= 9:  return date(d.year, 9, 30)
        return date(d.year, 12, 31)

    quarters = []
    d = window_start
    while d <= today:
        end = min(_quarter_end(d), today)
        quarters.append((d.strftime('%Y-%m-%d'), end.strftime('%Y-%m-%d')))
        d = _quarter_end(d) + timedelta(days=1)

    print(f'📅 Scraping {len(quarters)} quarter(s): {quarters[0][0]} → {quarters[-1][1]}')
    for i, (q_since, q_until) in enumerate(quarters):
        tweets = run_search_scrape(f'Latest {q_since}→{q_until}', q_since, q_until, sort='Latest')
        all_raw_tweets.extend(tweets)
        if i < len(quarters) - 1:
            time.sleep(65)

print(f'\n📊 After Latest search: {len(all_raw_tweets)} total tweets')


🔍 [Q2 2025] Search (Latest) from 2025-04-01 to 2025-06-30...


[apify.scweet runId:BXDAI23Smnf0CCfVj] -> Status: RUNNING, Message: 
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:11.237Z ACTOR: Pulling container image of build m2fVRVY0Rjskb9ddt from registry.
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:11.241Z ACTOR: Creating container.
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:11.307Z ACTOR: Starting container.
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:14.980Z [apify] INFO  Initializing Actor...
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:14.983Z [apify] INFO  System info ({"apify_sdk_version": "2.1.0", "apify_client_version": "1.8.1", "crawlee_version": "0.4.5", "python_version": "3.12.13", "os": "linux"})
[apify.scweet runId:BXDAI23Smnf0CCfVj] -> 2026-04-21T14:36:27.336Z [apify] INFO  [2026-04-21 14:36:27.336518+00:00] [worker=main] WARNING: You've exhausted your free daily tweet quota (1000 per day). Upgrade for more: https://apify.com/pricing?fpr=yhdrb
[apify.scweet r

  ✅ Got 0 tweets
🔍 [Q3 2025] Search (Latest) from 2025-07-01 to 2025-09-30...


[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> Status: RUNNING, Message: 
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:42.352Z ACTOR: Pulling container image of build m2fVRVY0Rjskb9ddt from registry.
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:47.312Z ACTOR: Creating container.
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:47.494Z ACTOR: Starting container.
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:50.580Z [apify] INFO  Initializing Actor...
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:50.583Z [apify] INFO  System info ({"apify_sdk_version": "2.1.0", "apify_client_version": "1.8.1", "crawlee_version": "0.4.5", "python_version": "3.12.13", "os": "linux"})
[apify.scweet runId:GqF7WfVzW5yXHlWUe] -> 2026-04-21T14:37:52.072Z [apify] INFO  [2026-04-21 14:37:52.072407+00:00] [worker=main] WARNING: You've exhausted your free daily tweet quota (1000 per day). Upgrade for more: https://apify.com/pricing?fpr=yhdrb
[apify.scweet r

  ✅ Got 0 tweets

📊 After Latest search: 0 total tweets


In [25]:
# =============================================================
# STRATEGY 2: Search (Top) — high-engagement tweets, full range to today
# =============================================================
time.sleep(65)

today_str = date.today().strftime('%Y-%m-%d')
tweets_top = run_search_scrape(
    f'Top {DEFAULT_START}→{today_str}',
    DEFAULT_START.strftime('%Y-%m-%d'),
    today_str,
    sort='Top',
)
all_raw_tweets.extend(tweets_top)

print(f'\n📊 After Top search: {len(all_raw_tweets)} total tweets (before dedup)')


🔍 [Top (Apr-Sep 2025)] Search (Top) from 2025-04-01 to 2025-09-30...


[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> Status: RUNNING, Message: 
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:07.625Z ACTOR: Pulling container image of build m2fVRVY0Rjskb9ddt from registry.
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:07.630Z ACTOR: Creating container.
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:07.712Z ACTOR: Starting container.
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:10.325Z [apify] INFO  Initializing Actor...
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:10.327Z [apify] INFO  System info ({"apify_sdk_version": "2.1.0", "apify_client_version": "1.8.1", "crawlee_version": "0.4.5", "python_version": "3.12.13", "os": "linux"})
[apify.scweet runId:1uSAJLTcoSXFGdjIb] -> 2026-04-21T14:39:12.410Z [apify] INFO  [2026-04-21 14:39:12.409756+00:00] [worker=main] WARNING: You've exhausted your free daily tweet quota (1000 per day). Upgrade for more: https://apify.com/pricing?fpr=yhdrb
[apify.scweet r

  ✅ Got 0 tweets

📊 After Top search: 0 total tweets (before dedup)


In [26]:
# =============================================================
# STRATEGY 3: Profile timelines — @PiggyVest and @usepocket
# =============================================================
time.sleep(65)

tweets_profiles = run_profile_scrape('Product profiles', product_profiles)
all_raw_tweets.extend(tweets_profiles)

print(f'\n📊 After profile scrape: {len(all_raw_tweets)} total tweets (before dedup)')


👤 [Product profiles] Profile timelines: ['@PiggyVest', '@usepocket']...


[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> Status: RUNNING, Message: 
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:27.345Z ACTOR: Pulling container image of build m2fVRVY0Rjskb9ddt from registry.
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:27.360Z ACTOR: Creating container.
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:27.449Z ACTOR: Starting container.
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:30.133Z [apify] INFO  Initializing Actor...
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:30.136Z [apify] INFO  System info ({"apify_sdk_version": "2.1.0", "apify_client_version": "1.8.1", "crawlee_version": "0.4.5", "python_version": "3.12.13", "os": "linux"})
[apify.scweet runId:NtJ4XyzCyaz6j0xSI] -> 2026-04-21T14:40:31.859Z [apify] INFO  [2026-04-21 14:40:31.844456+00:00] [worker=main] WARNING: You've exhausted your free daily tweet quota (1000 per day). Upgrade for more: https://apify.com/pricing?fpr=yhdrb
[apify.scweet r

  ✅ Got 0 tweets

📊 After profile scrape: 0 total tweets (before dedup)


In [27]:
# =============================================================
# DEDUPLICATE & SAVE — APPEND to existing data/raw_tweets_all.csv
# =============================================================
raw_csv = DATA_DIR / 'raw_tweets_all.csv'
new_df = pd.DataFrame(all_raw_tweets)
print(f'New tweets scraped: {len(new_df)}')

if raw_csv.exists():
    existing_df = pd.read_csv(raw_csv)
    print(f'Existing tweets on disk: {len(existing_df)}')
    combined_df = pd.concat([existing_df, new_df], ignore_index=True)
else:
    print('No existing file — starting fresh')
    existing_df = pd.DataFrame()
    combined_df = new_df

combined_df = combined_df.drop_duplicates(subset='tweet_id').reset_index(drop=True)

print(f'\n📊 Breakdown by scrape source:')
print(combined_df['scrape_source'].value_counts())

combined_df.to_csv(raw_csv, index=False)

print(f'\nBefore: {len(existing_df)} | New unique: {len(combined_df) - len(existing_df)}')
print(f'✅ Total unique tweets saved: {len(combined_df)}')

# ── Timestamp coverage diagnostic — CRITICAL for the dashboard ─────────────
ts_col = combined_df['posted_at_utc'] if 'posted_at_utc' in combined_df.columns else combined_df.get('created_at', pd.Series(dtype=str))
ts_filled = ts_col.astype(str).replace('nan', '').ne('').sum()
pct = 100 * ts_filled / max(len(combined_df), 1)
print(f'\n⏱  Timestamp coverage: {ts_filled}/{len(combined_df)} ({pct:.1f}%)')
if ts_filled < len(combined_df):
    missing = combined_df[ts_col.astype(str).replace('nan', '').eq('')].head(5)
    print('\n⚠️  Sample tweets WITHOUT a timestamp — inspect these (Scweet may have changed field names):')
    for _, t in missing.iterrows():
        print(f"   id={t['tweet_id']} source={t.get('scrape_source','')} text={t['tweet_text'][:60]!r}")

# ── Tweet date range & daily cadence (what the dashboard will show) ────────
if 'posted_date' in combined_df.columns and combined_df['posted_date'].astype(str).ne('').any():
    dates = combined_df['posted_date'].astype(str).replace('', pd.NA).dropna()
    print(f'\n📅 Date range (Lagos local): {dates.min()} → {dates.max()}')
    print(f'📊 Tweets per day (top 10 busiest days):')
    print(combined_df['posted_date'].value_counts().head(10).to_string())

print(f'\n--- Sample tweets ---')
for _, t in combined_df.sample(min(5, len(combined_df))).iterrows():
    print(f"  @{t['author_username']}: {t['tweet_text'][:100]}...")
    print(f"    📅 {t.get('posted_at_lagos', t.get('created_at', ''))} | "
          f"Source: {t['scrape_source']} | ❤️ {t['favorite_count']}\n")


New tweets scraped: 0
Existing tweets on disk: 1000

📊 Breakdown by scrape source:
scrape_source
search_latest    1000
Name: count, dtype: int64

Before: 1000 | New unique: 0
✅ Total unique tweets saved: 1000

⏱  Timestamp coverage: 1000/1000 (100.0%)

📅 Date range (Lagos local): 2025-05-09 → 2025-07-01
📊 Tweets per day (top 10 busiest days):
posted_date
2025-06-07    200
2025-06-30    196
2025-06-27    108
2025-06-21     94
2025-06-28     81
2025-05-10     76
2025-06-22     75
2025-05-09     63
2025-05-11     49
2025-06-20     31

--- Sample tweets ---
  @doyin_hd: @Callme_NNENNA Piggyvest for better interest rate.
Cowrywise for mutual funds and building disciplin...
    📅 2025-05-09T21:26:07+01:00 | Source: search_latest | ❤️ 1

  @notnaskii: Today's set is probably my favorite thing I've styled this year, then the piggyvest one...
    📅 2025-05-11T20:06:50+01:00 | Source: search_latest | ❤️ 1

  @Jacynthaekeh3: @MarinatedTurks Take my piggyvest pin then 🥺...
    📅 2025-06-27T18:15:3

### 💡 Didn't hit 1,000 tweets?
Re-run the strategies above on another day with different date ranges — dedup handles the overlap. Example below.

In [28]:
# OPTIONAL: run next day to add more tweets
# time.sleep(65)
# tweets_extra = run_search_scrape('Mid 2025', '2025-06-01', '2025-09-30', sort='Latest', max_items=1000)
#
# prev_df = pd.read_csv(DATA_DIR / 'raw_tweets_all.csv')
# extra_df = pd.DataFrame(tweets_extra)
# combined = pd.concat([prev_df, extra_df]).drop_duplicates(subset='tweet_id').reset_index(drop=True)
# combined.to_csv(DATA_DIR / 'raw_tweets_all.csv', index=False)
# print(f'✅ Now have {len(combined)} total unique tweets!')


## Step 5: Clean & filter

Preserves `author_username`, `author_name`, `author_followers`, and engagement metrics.

In [29]:
def clean_tweets(raw_tweets):
    """Dedupe + text-clean. Also **upgrades old-schema rows**: if a tweet came
    from an older CSV that only had `created_at` (space-separated, no TZ), we
    re-derive the full timestamp column set so the dashboard sees uniform data."""
    cleaned, seen_ids = [], set()
    for tweet in raw_tweets:
        tweet_id = str(tweet.get('tweet_id', ''))
        text = tweet.get('tweet_text', '')
        if not tweet_id or tweet_id in seen_ids:
            continue
        seen_ids.add(tweet_id)
        text = re.sub(r'http\S+', '', text.strip())
        text = re.sub(r'\s+', ' ', text).strip()
        if len(text) < 15:
            continue

        # Figure out the tweet's posted-at UTC datetime from whatever we have.
        posted_at_dt = (
            _parse_iso_utc(tweet.get('posted_at_utc'))
            or _parse_iso_utc(tweet.get('created_at'))
            or tweet_id_to_datetime_utc(tweet_id)
        )
        ts = timestamp_columns(posted_at_dt)

        cleaned.append({
            'tweet_id': tweet_id,
            'tweet_text': text,
            **ts,
            'created_at': ts['posted_at_utc'],  # legacy alias
            'collected_at': tweet.get('collected_at', ''),
            'author_username': tweet.get('author_username', ''),
            'author_name': tweet.get('author_name', ''),
            'author_followers': tweet.get('author_followers', 0),
            'favorite_count': tweet.get('favorite_count', 0),
            'retweet_count': tweet.get('retweet_count', 0),
            'reply_count': tweet.get('reply_count', 0),
            'tweet_url': tweet.get('tweet_url', ''),
            'scrape_source': tweet.get('scrape_source', ''),
        })
    print(f'Raw: {len(raw_tweets)} → Cleaned: {len(cleaned)} (removed {len(raw_tweets) - len(cleaned)})')
    # Coverage check — warn if anything lost its timestamp during cleaning
    with_ts = sum(1 for c in cleaned if c['posted_at_utc'])
    if with_ts < len(cleaned):
        print(f'⚠️  {len(cleaned) - with_ts} cleaned tweets have no timestamp')
    else:
        print(f'✅ All {len(cleaned)} cleaned tweets have ISO 8601 UTC timestamps (second precision)')
    return cleaned


def filter_tweets(cleaned_tweets):
    filtered = []
    for tweet in cleaned_tweets:
        text = tweet['tweet_text'].lower()
        if len(tweet['tweet_text']) < 30:
            continue
        if not any(kw in text for kw in brand_keywords):
            continue
        filtered.append(tweet)
    print(f'Cleaned: {len(cleaned_tweets)} → Filtered: {len(filtered)} (removed {len(cleaned_tweets) - len(filtered)})')
    return filtered

print('✅ clean_tweets() and filter_tweets() ready')


✅ clean_tweets() and filter_tweets() ready


In [30]:
# Clean + filter the full raw set
raw_df = pd.read_csv(DATA_DIR / 'raw_tweets_all.csv')
raw_tweets = raw_df.to_dict('records')
print(f'Loaded {len(raw_tweets)} raw tweets')

cleaned = clean_tweets(raw_tweets)
filtered = filter_tweets(cleaned)

all_df = pd.DataFrame(filtered)
all_df.to_csv(DATA_DIR / 'filtered_tweets_all.csv', index=False)

print(f'\n📊 Final filtered tweets: {len(all_df)}')
print(f'   Columns: {list(all_df.columns)}')
print(f'   Authors captured:      {(all_df["author_username"].astype(str) != "").sum()} / {len(all_df)}')
print(f'   UTC timestamps:        {(all_df["posted_at_utc"].astype(str) != "").sum()} / {len(all_df)}')
print(f'   Lagos local timestamps: {(all_df["posted_at_lagos"].astype(str) != "").sum()} / {len(all_df)}')

if (all_df['posted_at_utc'].astype(str) != '').any():
    dates = all_df[all_df['posted_at_utc'].astype(str) != '']['posted_at_utc'].sort_values()
    print(f'   Earliest tweet: {dates.iloc[0]}')
    print(f'   Latest tweet:   {dates.iloc[-1]}')

print('\n--- Sample tweets (showing Lagos local time) ---')
for _, t in all_df.sample(min(5, len(all_df))).iterrows():
    print(f"  @{t['author_username']}: {t['tweet_text'][:80]}...")
    print(f"    📅 {t['posted_at_lagos']} ({t['posted_dow']}) | "
          f"❤️ {t['favorite_count']} | 🔁 {t['retweet_count']}\n")


Loaded 1000 raw tweets
Raw: 1000 → Cleaned: 999 (removed 1)
✅ All 999 cleaned tweets have ISO 8601 UTC timestamps (second precision)
Cleaned: 999 → Filtered: 939 (removed 60)

📊 Final filtered tweets: 939
   Columns: ['tweet_id', 'tweet_text', 'posted_at_utc', 'posted_at_lagos', 'posted_date', 'posted_hour_utc', 'posted_hour_lagos', 'posted_dow', 'posted_week', 'created_at', 'collected_at', 'author_username', 'author_name', 'author_followers', 'favorite_count', 'retweet_count', 'reply_count', 'tweet_url', 'scrape_source']
   Authors captured:      939 / 939
   UTC timestamps:        939 / 939
   Lagos local timestamps: 939 / 939
   Earliest tweet: 2025-05-09T10:35:59Z
   Latest tweet:   2025-06-30T23:43:02Z

--- Sample tweets (showing Lagos local time) ---
  @carlaacarlson: Bankrate's 2025 Emergency Savings Report | Bankrate...
    📅 2025-06-28T07:57:47+01:00 (Saturday) | ❤️ 4 | 🔁 7

  @theajokeola_a: Omo this piggyvest thing isn't funny anymore @piggyvest @OdunEweniyi you guys ar...
 

## Step 6: Auto-label with Gemini — intent as aspect

In [31]:
def label_tweet(tweet_id, tweet_text, author_username='', author_name='', retries=3):
    assert gemini is not None, 'GEMINI_API_KEY missing — set it in ml/.env'

    prompt = f"""
You are an expert annotation system for aspect-based sentiment analysis of Nigerian fintech tweets.
Your job is to analyse tweets about PiggyTech products and return structured labels.

PRODUCTS:
- PiggyVest: savings, investments, withdrawals app
- PiggyVest_for_Business (PVB): business payments and collections
- Pocket: checkout and wallet app

PRODUCT ALIASES:
- PiggyVest: "piggyvest", "piggy", "@PiggyVest", "piggy vest", "safelock", "investify", "piggypoints", "flex dollar", "flex naira", "savings report", "piggyvest report"
- PiggyVest_for_Business: "pvb", "piggyvest for business", "@PVB", "bulk transfer", "salary disbursement"
- Pocket: "pocket app", "@usepocket", "usepocket", "pocket wallet", "pocket transfer", "payment link"

ASPECTS:
- Shared (any product): customer_support, app_performance, payments_transfers, notifications, trust_security, brand_perception, inquiry, suggestion
- PiggyVest only: savings_plans, withdrawals, investments, savings_report
- PiggyVest_for_Business only: merchant_onboarding, settlements, collections
- Pocket only: checkout_payments, wallet_funding, transfers

SPECIAL ASPECT RULES:
- Use "inquiry" as the aspect when the tweet is asking a question about the product
- Use "suggestion" as the aspect when the tweet is recommending a feature or improvement
- A tweet can have BOTH a regular aspect AND inquiry/suggestion if applicable
- inquiry and suggestion tweets still get a sentiment score

SENTIMENT OPTIONS: very_negative, slightly_negative, neutral, slightly_positive, very_positive

INTENT OPTIONS (for the intent field):
- opinion: user sharing their experience, viewpoint, or review
- complaint: user reporting a problem, bug, or expressing frustration
- inquiry: user asking a question
- suggestion: user recommending a feature or improvement
- spam: irrelevant, promotional, or bot-generated content

RULES:
1. Only detect products clearly mentioned
2. Only label aspects clearly discussed
3. overall_sentiment = general tone of whole tweet
4. aspect_sentiment = sentiment toward that specific aspect
5. One entry per aspect detected
6. If no PiggyTech product mentioned, return empty lists
7. intent = what the user is trying to do with this tweet
8. For inquiry/suggestion tweets, ALWAYS include inquiry or suggestion as one of the aspects

Tweet: "{tweet_text}"

Return ONLY valid JSON, no explanation, no markdown, no backticks:
{{
  "products_detected": ["product_name"],
  "overall_sentiment": "sentiment",
  "intent": "intent_label",
  "aspects": [
    {{
      "aspect_product": "product_name",
      "aspect": "aspect_name",
      "aspect_sentiment": "sentiment"
    }}
  ]
}}
"""

    for attempt in range(retries):
        try:
            response = gemini.generate_content(prompt)
            text = response.text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'):
                    text = text[4:]
            text = text.strip()
            result = json.loads(text)
            rows = []
            intent = result.get('intent', 'opinion')
            if intent not in intent_labels:
                intent = 'opinion'
            for aspect_info in result.get('aspects', []):
                rows.append({
                    'tweet_id': tweet_id,
                    'tweet_text': tweet_text,
                    'author_username': author_username,
                    'author_name': author_name,
                    'products_detected': result['products_detected'],
                    'overall_sentiment': result['overall_sentiment'],
                    'intent': intent,
                    'aspect_product': aspect_info['aspect_product'],
                    'aspect': aspect_info['aspect'],
                    'aspect_sentiment': aspect_info['aspect_sentiment'],
                })
            if not rows and result.get('products_detected'):
                rows.append({
                    'tweet_id': tweet_id,
                    'tweet_text': tweet_text,
                    'author_username': author_username,
                    'author_name': author_name,
                    'products_detected': result['products_detected'],
                    'overall_sentiment': result['overall_sentiment'],
                    'intent': intent,
                    'aspect_product': None,
                    'aspect': None,
                    'aspect_sentiment': None,
                })
            return rows
        except json.JSONDecodeError as e:
            print(f'❌ JSON parse error on tweet {tweet_id}: {e}')
            return []
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f'⚠️ Attempt {attempt + 1} failed, retrying in {wait}s... ({e})')
                time.sleep(wait)
            else:
                print(f'❌ All {retries} attempts failed for tweet {tweet_id}: {e}')
                return []

print('✅ label_tweet() ready')


✅ label_tweet() ready


In [32]:
# Label tweets — resumes from existing labelled_tweets_*.csv files
all_df = pd.read_csv(DATA_DIR / 'filtered_tweets_all.csv')

already_done = set()
existing_labels = pd.DataFrame()

for name in ['labelled_tweets_unified.csv', 'labelled_tweets_v2.csv', 'labelled_tweets_gemini.csv']:
    path = DATA_DIR / name
    if path.exists():
        temp = pd.read_csv(path)
        already_done.update(temp['tweet_id'].astype(str).unique())
        existing_labels = pd.concat([existing_labels, temp], ignore_index=True)
        print(f'Loaded {name}: {temp["tweet_id"].nunique()} unique tweets')

if len(existing_labels) > 0:
    existing_labels = existing_labels.drop_duplicates(subset=['tweet_id', 'aspect'], keep='last')

all_df['tweet_id'] = all_df['tweet_id'].astype(str)
remaining = [t for t in all_df.to_dict('records') if str(t['tweet_id']) not in already_done]

print(f'\nAlready labelled: {len(already_done)} tweets')
print(f'Remaining to label: {len(remaining)} tweets')

new_labels = []
checkpoint_path = DATA_DIR / 'labelled_tweets_checkpoint.csv'

for i, tweet in enumerate(remaining):
    rows = label_tweet(
        tweet_id=tweet['tweet_id'],
        tweet_text=tweet['tweet_text'],
        author_username=tweet.get('author_username', ''),
        author_name=tweet.get('author_name', ''),
    )
    new_labels.extend(rows)
    if (i + 1) % 20 == 0:
        print(f'  Labelled {i + 1}/{len(remaining)} remaining...')
        pd.DataFrame(new_labels).to_csv(checkpoint_path, index=False)
    time.sleep(1.0)

new_labels_df = pd.DataFrame(new_labels)
unified_df = pd.concat([existing_labels, new_labels_df], ignore_index=True)
unified_df = unified_df.drop_duplicates(subset=['tweet_id', 'aspect'], keep='last').reset_index(drop=True)

# Merge second-precision timestamps from filtered CSV so unified always has them
_ts_cols = ['posted_at_utc', 'posted_at_lagos', 'posted_date',
            'posted_hour_utc', 'posted_hour_lagos', 'posted_dow', 'posted_week', 'created_at']
_fdf = pd.read_csv(DATA_DIR / 'filtered_tweets_all.csv')
_ts_map = _fdf.set_index(_fdf['tweet_id'].astype(str))[[c for c in _ts_cols if c in _fdf.columns]]
unified_df['tweet_id'] = unified_df['tweet_id'].astype(str)
unified_df = unified_df.drop(columns=[c for c in _ts_cols if c in unified_df.columns], errors='ignore')
unified_df = unified_df.merge(_ts_map, left_on='tweet_id', right_index=True, how='left')

unified_df.to_csv(DATA_DIR / 'labelled_tweets_unified.csv', index=False)
unified_df.to_csv(DATA_DIR / 'labelled_tweets_v2.csv', index=False)

print(f'\n✅ Labelled {len(remaining)} new tweets → {len(new_labels_df)} new rows')
print(f'   Unified total: {len(unified_df)} rows ({unified_df["tweet_id"].nunique()} unique tweets)')
print(f'\n📊 Aspect distribution:')
print(unified_df['aspect'].value_counts())
print(f'\n📊 Intent distribution:')
print(unified_df['intent'].value_counts())


Loaded labelled_tweets_unified.csv: 852 unique tweets
Loaded labelled_tweets_v2.csv: 852 unique tweets

Already labelled: 852 tweets
Remaining to label: 87 tweets
⚠️ Attempt 1 failed, retrying in 1s... (429 Your project has exceeded its monthly spending cap. Please go to AI Studio at https://ai.studio/spend to manage your project spend cap.)
⚠️ Attempt 2 failed, retrying in 2s... (429 Your project has exceeded its monthly spending cap. Please go to AI Studio at https://ai.studio/spend to manage your project spend cap.)
❌ All 3 attempts failed for tweet tweet-1939816770845778377: 429 Your project has exceeded its monthly spending cap. Please go to AI Studio at https://ai.studio/spend to manage your project spend cap.
⚠️ Attempt 1 failed, retrying in 1s... (429 Your project has exceeded its monthly spending cap. Please go to AI Studio at https://ai.studio/spend to manage your project spend cap.)
⚠️ Attempt 2 failed, retrying in 2s... (429 Your project has exceeded its monthly spending ca

## Step 7: Quality check & train-ready split

In [33]:
df = pd.read_csv(DATA_DIR / 'labelled_tweets_unified.csv')

print(f'Total rows: {len(df)}')
print(f'Unique tweets: {df["tweet_id"].nunique()}')
print(f'\nSentiment distribution:')
print(df['overall_sentiment'].value_counts())
print(f'\nAspect distribution (inc. inquiry & suggestion):')
print(df['aspect'].value_counts())
print(f'\nIntent distribution:')
print(df['intent'].value_counts())
print(f'\nProduct distribution:')
print(df['products_detected'].value_counts())
print(f'\nAuthor coverage:')
print(f'  Tweets with author: {(df["author_username"] != "").sum()}')
print(f'  Unique authors: {df["author_username"].nunique()}')


Total rows: 1181
Unique tweets: 852

Sentiment distribution:
overall_sentiment
very_negative        367
neutral              264
very_positive        226
slightly_negative    221
slightly_positive    103
Name: count, dtype: int64

Aspect distribution (inc. inquiry & suggestion):
aspect
inquiry               175
savings_plans         166
brand_perception      154
withdrawals           153
customer_support      142
app_performance        84
trust_security         83
investments            69
payments_transfers     43
transfers              17
suggestion             16
notifications           8
wallet_funding          8
savings_report          8
Name: count, dtype: int64

Intent distribution:
intent
opinion       513
complaint     436
inquiry       197
suggestion     19
spam           16
Name: count, dtype: int64

Product distribution:
products_detected
['PiggyVest']                 1074
['Pocket']                      52
['PiggyVest', 'Pocket']         21
['Pocket', 'PiggyVest']         

### Filter & analyse by date

In [34]:
df = pd.read_csv(DATA_DIR / 'labelled_tweets_unified.csv')
filtered_df = pd.read_csv(DATA_DIR / 'filtered_tweets_all.csv')

# Backfill full timestamp suite onto the labelled CSV.
# Check for missing columns OR columns that are entirely NaN (previous partial backfill).
ts_cols = ['posted_at_utc', 'posted_at_lagos', 'posted_date',
           'posted_hour_utc', 'posted_hour_lagos', 'posted_dow', 'posted_week']
need_backfill = any(
    c not in df.columns or df[c].isna().any()
    for c in ts_cols
)
if need_backfill:
    df['tweet_id'] = df['tweet_id'].astype(str)
    ts_map = filtered_df.set_index(filtered_df['tweet_id'].astype(str))[ts_cols]
    # Drop stale/NaN timestamp columns before re-merging
    df = df.drop(columns=[c for c in ts_cols if c in df.columns], errors='ignore')
    df = df.merge(ts_map, left_on='tweet_id', right_index=True, how='left')
    df.to_csv(DATA_DIR / 'labelled_tweets_unified.csv', index=False)
    print(f'✅ Backfilled timestamp columns into labelled_tweets_unified.csv')
else:
    print('✅ Timestamp columns already present')

# Convenience datetime column for pandas groupby (not saved — just for this cell)
df['_posted_dt_lagos'] = pd.to_datetime(df['posted_at_lagos'], errors='coerce', utc=True).dt.tz_convert('Africa/Lagos')
df['_month']           = df['_posted_dt_lagos'].dt.to_period('M')

print(f'Total rows: {len(df)}')
print(f'Date range (Lagos): {df["posted_date"].min()} → {df["posted_date"].max()}')

print(f'\n📊 Tweets per month:')
print(df.groupby('_month')['tweet_id'].nunique())

print(f'\n📊 Sentiment by month:')
print(df.groupby(['_month', 'overall_sentiment'])['tweet_id'].nunique().unstack(fill_value=0))

print(f'\n📊 Tweets by hour-of-day (Lagos local) — spot peak complaint windows:')
print(df.groupby('posted_hour_lagos')['tweet_id'].nunique().sort_index())

print(f'\n📊 Tweets by day-of-week (Lagos local):')
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
print(df.groupby('posted_dow')['tweet_id'].nunique().reindex(dow_order).fillna(0).astype(int))

print(f'\n📊 Top aspects by month:')
print(df.dropna(subset=['aspect']).groupby(['_month', 'aspect']).size().unstack(fill_value=0))


Total rows: 1181
Date range (Lagos): nan → nan

📊 Tweets per month:
Series([], Freq: M, Name: tweet_id, dtype: int64)

📊 Sentiment by month:
Empty DataFrame
Columns: []
Index: []

📊 Tweets by hour-of-day (Lagos local) — spot peak complaint windows:
Series([], Name: tweet_id, dtype: int64)

📊 Tweets by day-of-week (Lagos local):
posted_dow
Monday       0
Tuesday      0
Wednesday    0
Thursday     0
Friday       0
Saturday     0
Sunday       0
Name: tweet_id, dtype: int64

📊 Top aspects by month:
Empty DataFrame
Columns: []
Index: []


/var/folders/qv/zd8l1bzs3w51ltqt9n6838vr0000gn/T/ipykernel_32532/3797246394.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['_month']           = df['_posted_dt_lagos'].dt.to_period('M')


In [35]:
# Prepare training data — one row per aspect
df_train = df.dropna(subset=['aspect', 'aspect_sentiment']).copy()
df_train['label'] = df_train['aspect_sentiment']
df_train['input_text'] = df_train['tweet_text'] + ' [ASPECT] ' + df_train['aspect']

print(f'Training rows: {len(df_train)}')
print('\nLabel distribution:')
print(df_train['label'].value_counts())

label2id = {
    'very_negative': 0,
    'slightly_negative': 1,
    'neutral': 2,
    'slightly_positive': 3,
    'very_positive': 4,
}
id2label = {v: k for k, v in label2id.items()}

df_train['label_id'] = df_train['label'].map(label2id)
df_train = df_train.dropna(subset=['label_id'])
df_train['label_id'] = df_train['label_id'].astype(int)

print(f'\nFinal training rows: {len(df_train)}')
df_train.to_csv(DATA_DIR / 'train_ready_unified.csv', index=False)
print('✅ Saved train_ready_unified.csv')


Training rows: 1126

Label distribution:
label
very_negative        348
very_positive        241
neutral              238
slightly_negative    208
slightly_positive     91
Name: count, dtype: int64

Final training rows: 1126
✅ Saved train_ready_unified.csv


## Step 8: Fine-tune classification model

> **Note on Apple Silicon:** the HuggingFace `Trainer` auto-detects and uses MPS.
> We set `fp16=False` and `use_cpu=False` — MPS is fp32 only, and mixed precision on MPS is still flaky.
> On an M1/M2 expect roughly 5-15 min per epoch on ~2-5k rows; bigger datasets scale linearly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# Re-apply Python 3.14 fingerprint patch now that datasets is imported
try:
    import datasets.fingerprint as _hf_fp
    if not getattr(_hf_fp.generate_fingerprint, '_patched', False):
        _orig_gen_fp = _hf_fp.generate_fingerprint
        def _safe_generate_fingerprint(dataset):
            try:
                return _orig_gen_fp(dataset)
            except TypeError:
                import hashlib, uuid
                return hashlib.md5(str(uuid.uuid4()).encode()).hexdigest()
        _safe_generate_fingerprint._patched = True
        _hf_fp.generate_fingerprint = _safe_generate_fingerprint
        print('🔧 Python 3.14 datasets fingerprint patch applied')
except Exception:
    pass

df_train = pd.read_csv(DATA_DIR / 'train_ready_unified.csv')

train_df, val_df = train_test_split(
    df_train, test_size=0.2, random_state=42, stratify=df_train['label_id']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

train_dataset = Dataset.from_pandas(train_df[['input_text', 'label_id']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['input_text', 'label_id']].reset_index(drop=True))

model_name = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column('label_id', 'labels')
val_dataset   = val_dataset.rename_column('label_id', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Data ready for training')


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1':       f1_score(labels, predictions, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / 'classification'),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=str(CHECKPOINT_DIR / 'logs_classification'),
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    fp16=False,     # MPS is fp32 only
    use_cpu=False,  # let Trainer pick MPS/CUDA
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete')


In [ ]:
save_path = MODEL_DIR / 'final_model_v2'
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f'✅ Classification model saved to {save_path}')


## Step 9: Fine-tune regression model

In [ ]:
import torch.nn as nn
from transformers import AutoModel

label2score = {
    'very_negative': 1.0,
    'slightly_negative': 2.0,
    'neutral': 3.0,
    'slightly_positive': 4.0,
    'very_positive': 5.0,
}

df_train = pd.read_csv(DATA_DIR / 'train_ready_unified.csv')
df_train['score'] = df_train['label'].map(label2score)

train_df, val_df = train_test_split(
    df_train, test_size=0.2, random_state=42, stratify=df_train['label_id']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = Dataset.from_pandas(train_df[['input_text', 'score']].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[['input_text', 'score']].reset_index(drop=True))

def tokenize(batch):
    return tokenizer(batch['input_text'], truncation=True, padding='max_length', max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column('score', 'labels')
val_dataset   = val_dataset.rename_column('score', 'labels')

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print('✅ Regression data ready')


In [ ]:
class RoBERTaRegression(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.regressor(cls_output).squeeze(-1)
        loss = None
        if labels is not None:
            loss = nn.MSELoss()(logits, labels.float())
        return (loss, logits) if loss is not None else logits

reg_model = RoBERTaRegression(model_name)

def compute_regression_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    mae  = mean_absolute_error(labels, predictions)
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    pred_classes  = np.clip(np.round(predictions), 1, 5)
    label_classes = np.clip(np.round(labels), 1, 5)
    accuracy = float(np.mean(pred_classes == label_classes))
    return {'mae': mae, 'rmse': rmse, 'accuracy': accuracy}

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / 'regression'),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir=str(CHECKPOINT_DIR / 'logs_regression'),
    logging_steps=20,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='mae',
    greater_is_better=False,
    report_to='none',
    fp16=False,
    use_cpu=False,
)

trainer = Trainer(
    model=reg_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_regression_metrics,
)

print('🚀 Starting regression training...')
trainer.train()
print('✅ Regression training complete')


In [ ]:
save_path = MODEL_DIR / 'regression_model_v2'
trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f'✅ Regression model saved to {save_path}')


## Step 10: Export for Label Studio (optional)

In [ ]:
labelled_df = pd.read_csv(DATA_DIR / 'labelled_tweets_unified.csv')
grouped = labelled_df.groupby('tweet_id')

ls_tasks = []
for tweet_id, group in grouped:
    first = group.iloc[0]
    products_detected = first['products_detected']
    ls_tasks.append({
        'data': {
            'tweet_id': str(tweet_id),
            'tweet_text': first['tweet_text'],
            'author_username': first.get('author_username', ''),
            'intent': first.get('intent', 'opinion'),
        },
        'predictions': [{
            'model_version': 'gemini-2.5-flash-v2',
            'result': [
                {
                    'from_name': 'overall_sentiment',
                    'to_name': 'tweet_text',
                    'type': 'choices',
                    'value': {'choices': [first['overall_sentiment']]},
                },
                {
                    'from_name': 'products_detected',
                    'to_name': 'tweet_text',
                    'type': 'choices',
                    'value': {
                        'choices': eval(products_detected) if isinstance(products_detected, str) else products_detected
                    },
                },
            ],
        }],
    })

out_path = DATA_DIR / 'label_studio_preannotated_v2.json'
with open(out_path, 'w') as f:
    json.dump(ls_tasks, f, indent=2)

print(f'✅ Exported {len(ls_tasks)} tasks → {out_path}')
